<a href="https://colab.research.google.com/github/berthauser/dmeyf2026/blob/main/src/PredFinal/EntregaFinal/719_final_gerencial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6 WorkFlow Gerencial, futuro=SEP

### 6.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán enriquecer

#### 6.2  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab
*   Bajar el **dataset_historico** al Google Drive y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dm"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dm"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"
descargar  "gerencial_competencia_2026.csv.gz"


## 6.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Thu Sep 17 10:55:39 PM 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671287,35.9,1473300,78.7,1473300,78.7
Vcells,1242646,9.5,8388608,64.0,1978689,15.1


In [3]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘R.utils’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘R.oo’, ‘R.methodsS3’


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach,

#### Parametros
Si es gerente, no cambie nada
<br>Si es Analista, cambie el nombre del dataset

In [4]:
PARAM <- list()
PARAM$semilla_primigenia <- 100043

# 7190/7191 usaron ties random, 7290/7291 usaron average.
# Numero nuevo porque los archivos de Kaggle se llaman
# KA<experimento>_<envios>.csv y los anteriores ya estan subidos.
PARAM$experimento <- 7390
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

#### Carpeta del Experimento

In [5]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 6.3.1   Preprocesamiento del dataset

#### 6.3.1.1  DT incorporar dataset

In [6]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 6.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [7]:
dataset[ foto_mes==202006, internet:=NA]
dataset[ foto_mes==202006, mrentabilidad:=NA]
dataset[ foto_mes==202006, mrentabilidad_annual:=NA]
dataset[ foto_mes==202006, mcomisiones:=NA]
dataset[ foto_mes==202006, mactivos_margen:=NA]
dataset[ foto_mes==202006, mpasivos_margen:=NA]
dataset[ foto_mes==202006, mcuentas_saldo:=NA]
dataset[ foto_mes==202006, ctarjeta_visa_transacciones:=NA]
dataset[ foto_mes==202006, mtarjeta_visa_consumo:=NA]
dataset[ foto_mes==202006, mtarjeta_master_consumo:=NA]
dataset[ foto_mes==202006, ccallcenter_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]

#### 6.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, quizas ajustando por IPC ...
<br>Esta parte podrá ser abordada por todos los Analistas y también la Gerenciapero se decide pedagogicamente no incluirla en esta primer version para reducir la carga cognitiva

In [ ]:
# Problema #02  Data Drifting  ---  RESUELTO, pero mas abajo.
#
# La recomendacion adoptada es rank_cero_fijo (Grupo B), que gana
# 10/10 semillas contra "estandarizar" (Grupo A).
#
# El bloque NO va aca: corre al final del FE intra-mes. Motivo:
# rank_cero_fijo renombra <campo> a <campo>_rank y borra el
# original. Si corriera en esta posicion, el guard
#   atributos_presentes( c("mpayroll", "cliente_edad") )
# daria FALSE y mpayroll_sobre_edad no se crearia, sin error ni
# warning. Corriendo despues, esa variable se calcula con valores
# crudos y luego se rankea junto con el resto.
#
# Desvio del orden original del workflow, documentado a proposito.


#### 6.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [8]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

# ---------------------------------------------------------------
#  FE intra-mes  ---  Problema #03, Experimento 2 del Grupo B
#
#  Receta: flags binarios de tenencia + Z-Score por mes sobre las
#  monetarias ANTES de calcular los ratios, sin inversion
#  redundante (si existe zA/zB no se crea zB/zA).
#
#  Evidencia: Wilcoxon Exp_2 vs Baseline, W = 1.00, p = 0.0039
#  sobre 10 semillas; mediana de ganancia 33.351 -> 34.544.
#
#  El set exacto de variables no esta publicado: la tabla del
#  Grupo B es de importancia, no de definicion. Se reconstruyen
#  los ratios nombrados alli, leyendo la convencion del sufijo
#  "_z": los ratios contra cliente_edad usan el valor crudo, los
#  ratios entre monetarias usan los Z-Score.
#
#  Este bloque corre ANTES del Data Drifting de mas abajo, para
#  que los Z-Score se calculen sobre los valores originales y no
#  sobre los rangos.
# ---------------------------------------------------------------

# 1) flags binarios de tenencia
for (campo in c("cpayroll_trx", "mcuenta_corriente",
                "mprestamos_personales", "Visa_mpagominimo")) {
  if (campo %in% colnames(dataset)) {
    dataset[, paste0("flag_", campo) :=
      as.integer(!is.na(get(campo)) & get(campo) != 0)]
  }
}

# 2) Z-Score por mes, previo a los ratios
campos_zscore <- c("mcaja_ahorro", "mcuentas_saldo", "mcuenta_corriente",
                   "mprestamos_personales", "mtarjeta_visa_consumo",
                   "ctrx_quarter", "cpayroll_trx")

for (campo in intersect(campos_zscore, colnames(dataset))) {
  dataset[, paste0("z_", campo) :=
    (get(campo) - mean(get(campo), na.rm = TRUE)) /
     sd(get(campo), na.rm = TRUE),
    by = list(foto_mes)]
}

# 3) ratios
crear_ratio <- function(nombre, num, den) {
  if (all(c(num, den) %in% colnames(dataset))) {
    v <- dataset[[num]] / dataset[[den]]
    v[!is.finite(v)] <- NA_real_   # un denominador z cercano a cero da Inf
    dataset[, (nombre) := v]
  } else {
    cat("salteo", nombre, "- falta alguna columna\n")
  }
}

# contra cliente_edad: valor crudo, sin sufijo _z
crear_ratio("ratio_ctrx_quarter_sobre_cliente_edad", "ctrx_quarter", "cliente_edad")
crear_ratio("ratio_cpayroll_trx_sobre_cliente_edad", "cpayroll_trx", "cliente_edad")
crear_ratio("ratio_mcaja_ahorro_sobre_cliente_edad", "mcaja_ahorro", "cliente_edad")

# entre monetarias: sobre los Z-Score, sufijo _z
crear_ratio("ratio_mcaja_ahorro_sobre_mprestamos_personales_z",
            "z_mcaja_ahorro", "z_mprestamos_personales")
crear_ratio("ratio_mcuentas_saldo_sobre_mprestamos_personales_z",
            "z_mcuentas_saldo", "z_mprestamos_personales")
crear_ratio("ratio_mcuenta_corriente_sobre_mprestamos_personales_z",
            "z_mcuenta_corriente", "z_mprestamos_personales")
crear_ratio("ratio_ctrx_quarter_sobre_mprestamos_personales_z",
            "z_ctrx_quarter", "z_mprestamos_personales")
crear_ratio("ratio_cpayroll_trx_sobre_mcuenta_corriente_z",
            "z_cpayroll_trx", "z_mcuenta_corriente")

cat("FE Exp2: el dataset quedo con", ncol(dataset), "columnas\n")

# ---------------------------------------------------------------
#  DR  Data Drifting  ---  Problema #02
#  Recomendacion adoptada: rank_cero_fijo (Grupo B).
#  Gana 10/10 semillas contra "estandarizar", que es lo que
#  recomienda el Grupo A.
#
#  Ver la nota en la seccion 6.3.1.3 DR sobre por que este bloque
#  corre aca y no alla.
# ---------------------------------------------------------------

campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

#  DESEMPATE DETERMINISTA  ---  version final, 2026-09-17
#
#  El codigo del Grupo B usa ties.method = "random", que consume azar
#  del RNG de R y no hay ningun set.seed antes de este bloque.
#  Medido: dos corridas del MISMO notebook (7190 y 7191) dieron 20 AUC
#  distintos y el ganador del Grid Search se dio vuelta de
#  (256, 64, 1228) a (128, 64, 350). Eso rompe el PDF seccion 5, que
#  exige regenerar el submit exacto.
#
#  PRIMER INTENTO: "average". Reproducible -- 7290 y 7291 dieron
#  (256, 64, 246) identico -- pero caro. Un 2x2 sobre semillas x
#  desempate, midiendo la media de los 11 cortes en el Public:
#
#                      random    average
#       1 semilla      20.631     18.072
#      10 semillas     21.161     17.693
#
#  El desempate vale -3.01 y el semillerio +0.08. "average" colapsa
#  los valores empatados en UN SOLO rango y el arbol pierde la
#  posibilidad de partir entre ellos.
#
#  VERSION FINAL: "first". Reparte rangos distintos, como hacia
#  "random", y es determinista porque el orden de filas queda fijado
#  por el setorder(dataset, numero_de_cliente, foto_mes) de unas
#  lineas mas abajo, que corre antes de esta funcion.
#
#  No se usa "random" + set.seed porque frank corre dentro de
#  by = list(foto_mes) y data.table puede procesar los grupos en
#  paralelo: el orden de consumo del RNG dependeria de la cantidad de
#  hilos, y sembrar no alcanzaria.
#
#  Desvio declarado respecto del codigo del Grupo B.

drift_rank_cero_fijo <- function(campos_drift) {
  cat("inicio drift_rank_cero_fijo()\n")

  for (campo in campos_drift) {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "first") / .N, by = list(foto_mes)]
    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "first") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\nfin drift_rank_cero_fijo()\n")
}

setorder(dataset, numero_de_cliente, foto_mes)

PARAM$DR$metodo <- "rank_cero_fijo"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting\n"),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios)
)

FE Exp2: el dataset quedo con 53 columnas
inicio drift_rank_cero_fijo()
mrentabilidad  mrentabilidad_annual  mcomisiones  mactivos_margen  mpasivos_margen  mcuenta_corriente  mcaja_ahorro  mcuentas_saldo  mtarjeta_visa_consumo  mtarjeta_master_consumo  mprestamos_personales  mpayroll  Master_mpagominimo  Visa_mpagominimo  mpayroll_sobre_edad  
fin drift_rank_cero_fijo()


In [9]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

[1] "numero_de_cliente"                                    
 [2] "foto_mes"                                             
 [3] "internet"                                             
 [4] "cliente_edad"                                         
 [5] "cliente_antiguedad"                                   
 [6] "cproductos"                                           
 [7] "cdescubierto_preacordado"                             
 [8] "ctarjeta_visa_transacciones"                          
 [9] "cpayroll_trx"                                         
[10] "ccomisiones_mantenimiento"                            
[11] "ccallcenter_transacciones"                            
[12] "chomebanking_transacciones"                           
[13] "ctrx_quarter"                                         
[14] "Master_status"                                        
[15] "Master_fechaalta"                                     
[16] "Visa_status"                                          
[17] "Visa_fechaalta"                                       
[18] "clase_ternaria"                                       
[19] "kmes"                                                 
[20] "flag_cpayroll_trx"                                    
[21] "flag_mcuenta_corriente"                               
[22] "flag_mprestamos_personales"                           
[23] "flag_Visa_mpagominimo"                                
[24] "z_mcaja_ahorro"                                       
[25] "z_mcuentas_saldo"                                     
[26] "z_mcuenta_corriente"                                  
[27] "z_mprestamos_personales"                              
[28] "z_mtarjeta_visa_consumo"                              
[29] "z_ctrx_quarter"                                       
[30] "z_cpayroll_trx"                                       
[31] "ratio_ctrx_quarter_sobre_cliente_edad"                
[32] "ratio_cpayroll_trx_sobre_cliente_edad"                
[33] "ratio_mcaja_ahorro_sobre_cliente_edad"                
[34] "ratio_mcaja_ahorro_sobre_mprestamos_personales_z"     
[35] "ratio_mcuentas_saldo_sobre_mprestamos_personales_z"   
[36] "ratio_mcuenta_corriente_sobre_mprestamos_personales_z"
[37] "ratio_ctrx_quarter_sobre_mprestamos_personales_z"     
[38] "ratio_cpayroll_trx_sobre_mcuenta_corriente_z"         
[39] "mrentabilidad_rank"                                   
[40] "mrentabilidad_annual_rank"                            
[41] "mcomisiones_rank"                                     
[42] "mactivos_margen_rank"                                 
[43] "mpasivos_margen_rank"                                 
[44] "mcuenta_corriente_rank"                               
[45] "mcaja_ahorro_rank"                                    
[46] "mcuentas_saldo_rank"                                  
[47] "mtarjeta_visa_consumo_rank"                           
[48] "mtarjeta_master_consumo_rank"                         
[49] "mprestamos_personales_rank"                           
[50] "mpayroll_rank"                                        
[51] "Master_mpagominimo_rank"                              
[52] "Visa_mpagominimo_rank"                                
[53] "mpayroll_sobre_edad_rank"

#### 6.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

Esto se mostrará unicamente a la *modalidad Analista Sr*

In [ ]:
# No se implementa Feature Engineering a partir de Random Forest

#### 6.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [10]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [11]:
ncol(dataset)
colnames(dataset)

[1] 253

[1] "numero_de_cliente"                                           
  [2] "foto_mes"                                                    
  [3] "internet"                                                    
  [4] "cliente_edad"                                                
  [5] "cliente_antiguedad"                                          
  [6] "cproductos"                                                  
  [7] "cdescubierto_preacordado"                                    
  [8] "ctarjeta_visa_transacciones"                                 
  [9] "cpayroll_trx"                                                
 [10] "ccomisiones_mantenimiento"                                   
 [11] "ccallcenter_transacciones"                                   
 [12] "chomebanking_transacciones"                                  
 [13] "ctrx_quarter"                                                
 [14] "Master_status"                                               
 [15] "Master_fechaalta"                                            
 [16] "Visa_status"                                                 
 [17] "Visa_fechaalta"                                              
 [18] "clase_ternaria"                                              
 [19] "kmes"                                                        
 [20] "flag_cpayroll_trx"                                           
 [21] "flag_mcuenta_corriente"                                      
 [22] "flag_mprestamos_personales"                                  
 [23] "flag_Visa_mpagominimo"                                       
 [24] "z_mcaja_ahorro"                                              
 [25] "z_mcuentas_saldo"                                            
 [26] "z_mcuenta_corriente"                                         
 [27] "z_mprestamos_personales"                                     
 [28] "z_mtarjeta_visa_consumo"                                     
 [29] "z_ctrx_quarter"                                              
 [30] "z_cpayroll_trx"                                              
 [31] "ratio_ctrx_quarter_sobre_cliente_edad"                       
 [32] "ratio_cpayroll_trx_sobre_cliente_edad"                       
 [33] "ratio_mcaja_ahorro_sobre_cliente_edad"                       
 [34] "ratio_mcaja_ahorro_sobre_mprestamos_personales_z"            
 [35] "ratio_mcuentas_saldo_sobre_mprestamos_personales_z"          
 [36] "ratio_mcuenta_corriente_sobre_mprestamos_personales_z"       
 [37] "ratio_ctrx_quarter_sobre_mprestamos_personales_z"            
 [38] "ratio_cpayroll_trx_sobre_mcuenta_corriente_z"                
 [39] "mrentabilidad_rank"                                          
 [40] "mrentabilidad_annual_rank"                                   
 [41] "mcomisiones_rank"                                            
 [42] "mactivos_margen_rank"                                        
 [43] "mpasivos_margen_rank"                                        
 [44] "mcuenta_corriente_rank"                                      
 [45] "mcaja_ahorro_rank"                                           
 [46] "mcuentas_saldo_rank"                                         
 [47] "mtarjeta_visa_consumo_rank"                                  
 [48] "mtarjeta_master_consumo_rank"                                
 [49] "mprestamos_personales_rank"                                  
 [50] "mpayroll_rank"                                               
 [51] "Master_mpagominimo_rank"                                     
 [52] "Visa_mpagominimo_rank"                                       
 [53] "mpayroll_sobre_edad_rank"                                    
 [54] "internet_lag1"                                               
 [55] "cliente_edad_lag1"                                           
 [56] "cliente_antiguedad_lag1"                                     
 [57] "cproductos_lag1"                                             
 [58] "cdescubierto_preacordado_lag1"                               


#### 6.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  nni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [ ]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 6.3.2 Modelado

#### 6.3.2.1 Training Strategy

Esta etapa de Workflow de  Training Strategy esta pensada para la *Modalidad Gerencial* que posee el dataset reducido de [202005, 202109]
<br> Si usted es un Analista, posee el periodo de [201901, 202109] y deberá experimentar en que meses le conviene experimentar

<br> A la *Modalidad Gerencial* no se le complicada la vida con el undersampling de los continua, por eso PARAM$trainingstrategy$training_pct <- 1.0
<br> Sin embargo, si usted es  *Analista SR* posee un dataset 50 veces ( filas x columnas) más grande que la *Modalidad Gerencial*  y por un tema de velocidad y experimentación más rápida puede llegar a necesitar activar el undersampling de la clase mayoritaria, a pesar de estar corriendo en Google Cloud.

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 202005, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 202005, 202106 ]  donde se consideran el 100% de los CONTINUA

In [12]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

# Problema #12 (Undersampling), recomendacion del Grupo B.
# Afecta SOLO al Grid Search: training_pct -> fold_train -> dtrain,
# y dtrain se usa unicamente dentro de Estimar_AUC_lightgbm(). El
# modelo final usa dfinal_train, sin undersampling.
# B midio: sin punto de ruptura hasta 0.01, la media mas alta de los
# cinco niveles, y 87% menos de computo. El 0.1 que recomienda el
# Grupo A resulto el PEOR de los cinco en su medicion.
PARAM$trainingstrategy$training_pct <- 0.08


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [13]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [14]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

Esta celda tarda en correr interminables 7 minutos en Colab
<br> ya que debe instalar la librería de LightGBM

In [15]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

# control de cordura: con training_pct bajo conviene ver cuantas
# filas quedaron para el Grid Search
cat("dtrain:", sum(dataset$fold_train), "filas de",
    sum(dataset$foto_mes %in% PARAM$trainingstrategy$training), "posibles\n")


Loading required package: lightgbm

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘lightgbm’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: lightgbm



dtrain: 15876 filas de 179449 posibles


In [16]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 13202

####  6.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [17]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  # Reproducibilidad exacta. Sin num_threads fijo y deterministic,
  # el orden de reduccion en punto flotante cambia con la cantidad
  # de hilos y la misma semilla da otro modelo en otra maquina.
  # El PDF seccion 5 exige que los profesores puedan regenerar el
  # submit exacto.
  num_threads= 2,
  deterministic= TRUE,
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [18]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  niter <- modelo_train$best_iter
  # hago espacio en la memoria
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [19]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_leaves= c(64, 128, 256, 512),
  min_data_in_leaf= c(64, 256, 512, 1024, 2048)
)

Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora 50 minutos en Colab
<br> lamento profundamente tal intolerable espera gerencial

In [20]:
# registro a registro calculo la AUC
tb_nueva[, c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD ),
  by=1:nrow(tb_nueva) ]

Thu Sep 17 11:09:16 PM 2026  64, 64 niter 519 AUC 0.944653155058178

Thu Sep 17 11:09:39 PM 2026  64, 256 niter 467 AUC 0.942727262869102

Thu Sep 17 11:10:30 PM 2026  64, 512 niter 1984 AUC 0.94528384137415

Thu Sep 17 11:10:58 PM 2026  64, 1024 niter 889 AUC 0.939786807204745

Thu Sep 17 11:11:30 PM 2026  64, 2048 niter 1159 AUC 0.935313011700663

Thu Sep 17 11:12:10 PM 2026  128, 64 niter 716 AUC 0.946207749113504

Thu Sep 17 11:12:29 PM 2026  128, 256 niter 467 AUC 0.942727262869102

Thu Sep 17 11:13:26 PM 2026  128, 512 niter 1984 AUC 0.94528384137415

Thu Sep 17 11:13:55 PM 2026  128, 1024 niter 889 AUC 0.939786807204745

Thu Sep 17 11:14:26 PM 2026  128, 2048 niter 1159 AUC 0.935313011700663

Thu Sep 17 11:15:07 PM 2026  256, 64 niter 462 AUC 0.94617304835334

Thu Sep 17 11:15:28 PM 2026  256, 256 niter 467 AUC 0.942727262869102

Thu Sep 17 11:16:26 PM 2026  256, 512 niter 1984 AUC 0.94528384137415

Thu Sep 17 11:16:53 PM 2026  256, 1024 niter 889 AUC 0.939786807204745

Thu Sep 

la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [21]:
tb_nueva

fwrite( tb_nueva,
  file= "tb_grid_search_01.txt",
  sep="\t",
  append= TRUE
)

num_leaves,min_data_in_leaf,AUC,num_iterations
<dbl>,<dbl>,<dbl>,<int>
64,64,0.9446532,519
64,256,0.9427273,467
64,512,0.9452838,1984
64,1024,0.9397868,889
64,2048,0.9353130,1159
128,64,0.9462077,716
128,256,0.9427273,467
128,512,0.9452838,1984
128,1024,0.9397868,889


In [22]:
setorder( tb_nueva, -AUC)  # ordeno DESCENDENTE por AUC
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC] # en la posicion 1 estan los mejores
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

$num_leaves
[1] 128

$min_data_in_leaf
[1] 64

$num_iterations
[1] 716

### 6.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en optimización de hiperparámetros

In [23]:
PARAM$trainingstrategy$final_train <- c( 202107,
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

[1] 192651

##### Final Training Hyperparameters

In [24]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)


# ---------------------------------------------------------------
#  Experimento propio  ---  lo exige el PDF seccion 6.15
#
#  SEMILLERIO: se entrenan N modelos identicos salvo la semilla y
#  se PROMEDIAN las probabilidades antes de cortar.
#
#  Por que este y no otro: tanto el experimento del Problema 01
#  como la conclusion del Problema 11 midieron lo mismo, que el
#  ruido de la semilla explica lo que parecia efecto del metodo.
#  En el Problema 01 el rango entre las medias de los cinco brazos
#  fue 0.77 y el desvio entre semillas dentro de un brazo 1.35, y
#  la mejor y la peor corrida de todo el experimento salieron de
#  la MISMA semilla (700021). Si la semilla domina, promediar
#  sobre semillas deberia rendir mas que cualquier eleccion de
#  metodo.
#
#  Semillas propias de Ernesto. Para una corrida de prueba, dejar
#  una sola y el notebook se comporta como el original.
# ---------------------------------------------------------------

PARAM$semillerio <- c(100043, 200063, 300089, 500069, 700021,
                      181219, 410341, 568723, 618347, 831781)


##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [25]:
# ---------------------------------------------------------------
#  Final Training  ---  semillerio
#  Se acumula la prediccion dentro del loop en vez de guardar los
#  N modelos, para no tener N boosters vivos en memoria.
#  dfinal_train se reusa entre iteraciones: se probo en el
#  Problema 01 con 7 semillas y free_raw_data=TRUE sin problemas.
# ---------------------------------------------------------------

PARAM$trainingstrategy$future <- c(202109)
dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]
mfuture <- data.matrix(dfuture[, campos_buenos, with= FALSE])

prediccion_acum <- rep(0.0, nrow(dfuture))

for (vsemilla in PARAM$semillerio) {

  cat(format(Sys.time(), "%X"), " semilla", vsemilla, "... ")

  param_semilla <- param_final
  param_semilla$seed <- vsemilla

  final_model <- lgb.train(
    data= dfinal_train,
    param= param_semilla,
    verbose= -100
  )

  prediccion_acum <- prediccion_acum + predict(final_model, mfuture)

  cat("listo\n")
}

# el promedio de las probabilidades es la prediccion del semillerio
prediccion <- prediccion_acum / length(PARAM$semillerio)

cat("semillerio: ", length(PARAM$semillerio), " modelos promediados\n")
cat("prob  min ", min(prediccion), "  media ", mean(prediccion),
    "  max ", max(prediccion), "\n")

# final_model quedo con el ULTIMO modelo del loop: es el que se
# graba a disco y del que sale la importancia de variables.


11:22:06 PM  semilla 100043 ... listo
11:24:51 PM  semilla 200063 ... listo
11:27:26 PM  semilla 300089 ... listo
11:30:00 PM  semilla 500069 ... listo
11:32:42 PM  semilla 700021 ... listo
11:35:19 PM  semilla 181219 ... listo
11:37:54 PM  semilla 410341 ... listo
11:40:30 PM  semilla 568723 ... listo
11:43:07 PM  semilla 618347 ... listo
11:45:39 PM  semilla 831781 ... listo
semillerio:  10  modelos promediados
prob  min  2.057539e-06   media  0.004084718   max  0.8606358 


In [26]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(final_model, "modelo.txt")

In [27]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(final_model))
archivo_importancia <- "impo.txt"

fwrite( tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

#### Scoring

Aplico el modelo final a los datos del futuro

In [28]:
PARAM$trainingstrategy$future <- c(202109)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

In [29]:
# La prediccion YA esta calculada: es el promedio del semillerio,
# hecho en el Final Training de mas arriba. Si aca se volviera a
# llamar a predict(final_model, ...) se pisaria ese promedio con
# la salida de un solo modelo, el ultimo del loop.

stopifnot( length(prediccion) == nrow(dfuture) )
cat("prediccion del semillerio:", length(prediccion), "filas\n")


prediccion del semillerio: 13242 filas


##### Tabla Prediccion

In [30]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion]

# grabo las probabilidad del modelo
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggle
<br>El notebook esta preparado para la Modalidad Gerencial, los analistas deben hacer cambios.


In [31]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

PARAM$kaggle$competencia <- "utn-2026-virtual-mgr"
PARAM$kaggle$cortes <- seq(800, 1300, by = 50)

# Llave de seguridad: en FALSE genera los CSV en ./kaggle/ pero NO
# los sube. Sirve para corridas de prueba sin gastar submits ni
# ensuciar el leaderboard.
PARAM$kaggle$submit <- TRUE

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  if( PARAM$kaggle$submit ) {
    Sys.sleep(30)
    salida <- system(linea, intern=TRUE) # el submit a Kaggle
    cat(salida, "\n")
  } else {
    cat("(submit desactivado) generado ", archivo_kaggle, "\n")
  }
}

In [32]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

Loading required package: yaml



In [33]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Thu Sep 17 11:55:44 PM 2026"